**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
%run /Workspace/Users/niteshsh34@gmail.com/Final_project_fmcg/1_codes/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://de-project-fmcg/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

# defining the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

Base Path:  s3://de-project-fmcg/orders
Landing Path:  s3://de-project-fmcg/orders/landing/
Processed Path:  s3://de-project-fmcg/orders/processed/


## Bronze

In [0]:
df = spark.read.options(header=True, inferSchema=True)\
    .csv(f"{landing_path}/*.csv")\
    .withColumn("read_timestamp", F.current_timestamp())\
    .withColumn("file_name",F.col("_metadata.file_name"))\
    .withColumn("file_size",F.col("_metadata.file_size"))\

print("Total Rows: ", df.count())
df.show(5)

Total Rows:  51810
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FOCT62720602|Tuesday, Septembe...|     ABC987|  25891301|     71.0|2026-07-19 05:21:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891502|    125.0|2026-07-19 05:21:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891403|    462.0|2026-07-19 05:21:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|    INVALID|  25891601|    133.0|2026-07-19 05:21:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891602|     79.0|2026-07-19 05:21:...|orders_2025_09_30...|    41446|
+------------

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

### Moving files from source to processed directory

In [0]:
# Get the LIST of all files available in the landing folder
files = dbutils.fs.ls(landing_path)
# Loop through each file in the landing folder
for file_info in files:
    # Move the current file from the landing folder to the processed/archive folder
    dbutils.fs.mv(
        file_info.path,                         # Source path 
        f"{processed_path}/{file_info.name}",   # Destination path 
        True                                    # True allows recursive move
    )

## Silver

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
display(df_orders.limit(20))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FJUL33320501,2025/07/01,789320,25891203,150.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33320501,2025/07/01,789320,25891301,46.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33320501,01-07-2025,789320,25891403,null,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33320501,"Tuesday, July 01, 2025",789320,25891201,354.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33320501,"Tuesday, July 01, 2025",789320,25891501,249.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33320501,01-07-2025,789320,25891301,46.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33401603,"Tuesday, July 01, 2025",789401,25891302,40.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33401603,"Tuesday, July 01, 2025",789401,25891502,133.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33401603,"Tuesday, July 01, 2025",789401,25891503,145.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33401603,"Tuesday, July 01, 2025",789401,25891203,429.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744


**Transformations**

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())

# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)
# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(
        F.col("order_placement_date"),
        r"^[A-Za-z]+,\s*",
        ""
    )
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd',' yyyy")
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-07-01|2025-11-30|
+----------+----------+



In [0]:
display(df_orders.limit(20))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FJUL33320501,2025-07-01,789320,25891203,150.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891301,46.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891201,354.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891501,249.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891302,40.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891502,133.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891503,145.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891203,429.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891201,461.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744
FJUL32101601,2025-07-01,789101,25891503,183.0,2026-07-19T05:21:08.336Z,orders_2025_07_01.csv,20744


**Join with products**

In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = (
    df_orders
    .join(df_products, on="product_id", how="inner")
    .select(
        df_orders["*"],                  # Select all columns from df_orders
        df_products["product_code"]      # Add product_code from df_products
    )
)

df_joined.show(5)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|        product_code|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|FJUL32202603|          2025-07-01|     789202|  25891103|    370.0|2026-07-19 05:21:...|orders_2025_07_01...|    20744|102628255d24304d6...|
|FJUL33321602|          2025-07-01|     789321|  25891602|     74.0|2026-07-19 05:21:...|orders_2025_07_01...|    20744|778c2a7aa27bfdb21...|
|FJUL34303602|          2025-07-01|     789303|  25891102|    317.0|2026-07-19 05:21:...|orders_2025_07_01...|    20744|e92c739a8d78cd6cb...|
|FJUL34622602|          2025-07-01|     789622|  25891402|    439.0|2026-07-19 05:21:...|orders_2025_07_01...|    20744|fe5a8036be4b9a787...|
|FJUL3

In [0]:
df_joined.groupBy(
    "order_placement_date",
    "order_id",
    "product_code",
    "customer_id"
).count().filter("count > 1").show()

+--------------------+--------+------------+-----------+-----+
|order_placement_date|order_id|product_code|customer_id|count|
+--------------------+--------+------------+-----------+-----+
+--------------------+--------+------------+-----------+-----+



In [0]:
df_joined = df_joined.dropDuplicates([
    "order_placement_date",
    "order_id",
    "product_code",
    "customer_id"
])

In [0]:
if not (spark.catalog.tableExists(silver_table)): # silver table not exist
    df_joined.write.format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(
        df_joined.alias("bronze"),
        """
        silver.order_placement_date = bronze.order_placement_date
        AND silver.order_id = bronze.order_id
        AND silver.product_code = bronze.product_code
        AND silver.customer_id = bronze.customer_id
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

## Gold

In [0]:
df_gold = spark.sql(f"""
    SELECT 
        order_id,
        order_placement_date AS date,
        customer_id AS customer_code,
        product_code,
        product_id,
        order_qty AS sold_quantity
    FROM {silver_table}
""")

df_gold.show(10)

+-------------+----------+-------------+--------------------+----------+-------------+
|     order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+-------------+----------+-------------+--------------------+----------+-------------+
|FNOV731501503|2025-11-29|       789501|fe5a8036be4b9a787...|  25891402|        357.0|
|FNOV730202501|2025-11-29|       789202|c68834ceaff15846b...|  25891303|         55.0|
|FNOV731603102|2025-11-28|       789603|e92c739a8d78cd6cb...|  25891102|        328.0|
|FNOV731201101|2025-11-28|       789201|e91ba9d665f90254d...|  25891101|        445.0|
|FNOV729201602|2025-11-27|       789201|778c2a7aa27bfdb21...|  25891602|        138.0|
|FNOV729102301|2025-11-27|       789102|3cab59f0592428527...|  25891301|         95.0|
|FNOV729420503|2025-11-27|       789420|e92c739a8d78cd6cb...|  25891102|        356.0|
|FNOV729221501|2025-11-27|       789221|77b6f538a9d0e0cf8...|  25891403|        257.0|
|FNOV726622401|2025-11-25|       789622|da6

In [0]:
df_gold.groupBy(
    "date",
    "order_id",
    "product_code",
    "customer_code"
).count() \
 .filter("count > 1") \
 .show()

+----+--------+------------+-------------+-----+
|date|order_id|product_code|customer_code|count|
+----+--------+------------+-------------+-----+
+----+--------+------------+-------------+-----+



In [0]:
df_gold = df_gold.dropDuplicates([
    "date",
    "order_id",
    "product_code",
    "customer_code"
])
df_gold.show(10)

+-------------+----------+-------------+--------------------+----------+-------------+
|     order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+-------------+----------+-------------+--------------------+----------+-------------+
|FNOV731501503|2025-11-29|       789501|fe5a8036be4b9a787...|  25891402|        357.0|
|FNOV730202501|2025-11-29|       789202|c68834ceaff15846b...|  25891303|         55.0|
|FNOV731603102|2025-11-28|       789603|e92c739a8d78cd6cb...|  25891102|        328.0|
|FNOV731201101|2025-11-28|       789201|e91ba9d665f90254d...|  25891101|        445.0|
|FNOV729201602|2025-11-27|       789201|778c2a7aa27bfdb21...|  25891602|        138.0|
|FNOV729102301|2025-11-27|       789102|3cab59f0592428527...|  25891301|         95.0|
|FNOV729420503|2025-11-27|       789420|e92c739a8d78cd6cb...|  25891102|        356.0|
|FNOV729221501|2025-11-27|       789221|77b6f538a9d0e0cf8...|  25891403|        257.0|
|FNOV726622401|2025-11-25|       789622|da6

In [0]:
if not spark.catalog.tableExists(gold_table):
    print("Creating New Table")
    df_gold.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .option("mergeSchema", "true") \
        .mode("overwrite") \
        .saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(
        df_gold.alias("gold"),
        """
        source.date = gold.date
        AND source.order_id = gold.order_id
        AND source.product_code = gold.product_code
        AND source.customer_code = gold.customer_code
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

## Merging with Parent company

- Note: We want data for monthly level but child data is on daily level

**Full Load**

In [0]:
df_child = spark.sql(f"""SELECT date,
                      product_code, 
                      customer_code, 
                      sold_quantity 
                      FROM {gold_table}""")
df_child.show(10)

+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-07-01|102628255d24304d6...|       789202|        370.0|
|2025-07-01|062f5574bbdf4386b...|       789721|        137.0|
|2025-07-01|c68834ceaff15846b...|       789420|         57.0|
|2025-07-02|ee1f7df9cf660ef02...|       999999|        105.0|
|2025-07-02|77b6f538a9d0e0cf8...|       789202|        317.0|
|2025-07-03|3cab59f0592428527...|       789521|         63.0|
|2025-07-03|3cab59f0592428527...|       789622|         84.0|
|2025-07-03|e91ba9d665f90254d...|       999999|        400.0|
|2025-07-04|e91ba9d665f90254d...|       789101|        311.0|
|2025-07-04|e92c739a8d78cd6cb...|       789201|        350.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
df_child.count()

40811

In [0]:
# since for parent the data is month wise so we will do the same for child also
df_monthly = (
    df_child
    .withColumn(        # Get month start date
        "month_start",
        F.trunc("date", "MM")
    )
    .groupBy(           # Group data at monthly level
        "month_start",
        "product_code",
        "customer_code"
    )
    .agg(              # Calculate total sold quantity
        F.sum("sold_quantity").alias("sold_quantity")
    )                  # Rename month_start to date
    .withColumnRenamed(
        "month_start",
        "date"
    )
)
df_monthly.show(5, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-07-01|102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268|789202       |2944.0       |
|2025-07-01|062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379|789721       |2040.0       |
|2025-07-01|c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|789420       |1125.0       |
|2025-07-01|ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387f09b0cae0abae|999999       |9438.0       |
|2025-07-01|77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9|789202       |5275.0       |
+----------+----------------------------------------------------------------+-------------+-------------+
only showing top 5 rows


In [0]:
df_monthly.count()

3060

In [0]:
gold_parent_delta = DeltaTable.forName(
    spark,
    f"{catalog}.{gold_schema}.fact_orders"
)

gold_parent_delta.alias("parent_gold").merge(
    df_monthly.alias("child_gold"),
    """
    parent_gold.date = child_gold.date
    AND parent_gold.product_code = child_gold.product_code
    AND parent_gold.customer_code = child_gold.customer_code
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]